# BigQuery 집계 마트에서 DuckDB 만들기

서비스 계정 대신 본인 Google 로그인으로 집계 마트를 조회합니다. CSV를 만들지 않습니다. 생성한 파일은 저장 시점 스냅샷입니다. 공개 가능한 집계만 다루세요. Colab은 데이터 준비용이며 앱 서버로 사용하지 않습니다.

In [3]:
%pip -q install google-cloud-bigquery==3.45.0 db-dtypes==1.7.1 duckdb==1.5.5

In [4]:
from google.colab import auth
auth.authenticate_user()

## 본인 설정
실행 프로젝트와 마트 전체 주소, 리전을 바꿉니다. 원천 bdai13-bigquery.tabformer를 직접 모두 다운로드하지 않습니다.

In [5]:
QUERY_PROJECT = 'bdai13-bigquery'
MART_TABLE = 'bdai13-bigquery.bdai13.mart_monthly_mcc'
LOCATION = 'asia-northeast3'
MAX_BYTES = 100_000_000

In [6]:
from google.cloud import bigquery
from datetime import date, datetime, timezone
from decimal import Decimal
import re

assert re.fullmatch(r'[a-z][a-z0-9-]{4,61}[a-z0-9]\.[A-Za-z_][A-Za-z0-9_]*\.[A-Za-z_][A-Za-z0-9_]*', MART_TABLE), '마트 주소를 바꾸세요'
client = bigquery.Client(project=QUERY_PROJECT)
sql = f"""SELECT tx_month, mcc, channel, amount_usd, txn_count,
                 fraud_count, fraud_labeled_count
          FROM `{MART_TABLE}`
          WHERE tx_month >= @start_date AND tx_month < @end_date
          LIMIT 50001"""
config = bigquery.QueryJobConfig(maximum_bytes_billed=MAX_BYTES, query_parameters=[
    bigquery.ScalarQueryParameter('start_date', 'DATE', date(2018,1,1)),
    bigquery.ScalarQueryParameter('end_date', 'DATE', date(2019,1,1))])
job = client.query(sql, job_config=config, location=LOCATION)
df = job.result(timeout=90).to_dataframe(create_bqstorage_client=False)
assert 0 < len(df) <= 50000, '마트가 비어 있거나 너무 큽니다'
assert not df[['tx_month','mcc','channel']].duplicated().any(), '마트 그레인 중복'
extracted_at = datetime.now(timezone.utc).isoformat()
display(df.head())
print('행 수:', len(df), '처리 바이트:', job.total_bytes_processed)

,tx_month,mcc,channel,amount_usd,txn_count,fraud_count,fraud_labeled_count
0,2018-08-01,5732,오프라인,5976.210000000,45,0,45
1,2018-08-01,6300,오프라인,67219.360000000,283,0,283
2,2018-08-01,3000,오프라인,13107.060000000,19,0,19
3,2018-08-01,8931,오프라인,3975.400000000,23,0,23
4,2018-08-01,5261,오프라인,4231.820000000,86,0,86


행 수: 1536 처리 바이트: 106578


## DuckDB로 저장하고 합계를 대조
기존 mart.duckdb가 있으면 이번 집계로 교체됩니다. 금액은 DECIMAL로 저장하며 임의 반올림하지 않습니다.

In [7]:
import duckdb
import pyarrow as pa
arrow = pa.Table.from_pandas(df, preserve_index=False)
with duckdb.connect('mart.duckdb') as con:
    con.register('source_result', arrow)
    con.execute("""CREATE OR REPLACE TABLE mart_monthly_mcc AS
        SELECT CAST(tx_month AS DATE) AS tx_month, CAST(mcc AS BIGINT) AS mcc,
               CAST(channel AS VARCHAR) AS channel,
               CAST(amount_usd AS DECIMAL(38,9)) AS amount_usd,
               CAST(txn_count AS BIGINT) AS txn_count,
               CAST(fraud_count AS BIGINT) AS fraud_count,
               CAST(fraud_labeled_count AS BIGINT) AS fraud_labeled_count
        FROM source_result""")
    con.execute('CREATE OR REPLACE TABLE snapshot_metadata (extracted_at VARCHAR, source_table VARCHAR)')
    con.execute('INSERT INTO snapshot_metadata VALUES (?, ?)', [extracted_at, MART_TABLE])
    actual = con.execute('SELECT COUNT(*), SUM(amount_usd), SUM(txn_count), SUM(fraud_count), SUM(fraud_labeled_count) FROM mart_monthly_mcc').fetchone()
expected = (len(df), sum(df.amount_usd, Decimal('0')), int(df.txn_count.sum()), int(df.fraud_count.sum()), int(df.fraud_labeled_count.sum()))
assert actual == expected, (actual, expected)
print('BigQuery 결과와 DuckDB의 행 수 및 합계가 같습니다:', actual)

BigQuery 결과와 DuckDB의 행 수 및 합계가 같습니다: (1536, Decimal('72187810.030000000'), 1694535, 2382, 1694535)


In [8]:
from google.colab import files
files.download('mart.duckdb')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Streamlit에서 사용
집계 파일을 앱 저장소에 넣고 Secrets의 `[data]` 아래 `backend = "duckdb"`, `snapshot = "mart.duckdb"`를 설정합니다. 생성 시각이 앱에 표시됩니다. 새 데이터가 필요하면 이 노트북을 다시 실행해 파일을 바꿉니다. BigQuery 접근 권한이 없으면 이 노트북도 그 권한을 우회하지 못합니다.